# Individualized Initial Knowledge

By default all students in a KC share one initial knowledge parameter `pi_know`. With `individual_initial_knowledge=True`, each student gets their own. This example covers:

1. Fitting per student `pi_know` with the default `CORRECTNESS_ONLY` strategy.
2. Reading the per student estimates from `summary()`.
3. What happens when predicting students that were not part of the fit.

See the [Initial Knowledge](../guide/initial_knowledge.rst) guide page for details.

In [1]:
import logging

import numpy as np
import pandas as pd
from cmdstanpy.utils import get_logger

# keep the notebook output readable, CmdStan progress is logged at INFO level
get_logger().setLevel(logging.WARNING)

## Simulate students who start at different levels

We simulate two cohorts with the same learning parameters but different initial knowledge, then combine them. The model is not told which cohort a student is in.

In [2]:
from stanbkt.utils import sim_simple_BKT

shared = {"learn": 0.1, "forget": 0.01, "guess": 0.2, "slip": 0.1}
novices = sim_simple_BKT(n_students=25, n_problems=12, prior=0.05, rng_seed=21, **shared)
experts = sim_simple_BKT(n_students=25, n_problems=12, prior=0.9, rng_seed=22, **shared)
novices["student_id"] = "novice_" + novices["student_id"]
experts["student_id"] = "expert_" + experts["student_id"]
data_df = pd.concat([novices, experts], ignore_index=True)
data_df.groupby(data_df["student_id"].str.split("_").str[0])["correct"].mean()

student_id
expert    0.846667
novice    0.526667
Name: correct, dtype: float64

In [3]:
from stanbkt.fits import MCMCFitOptions
from stanbkt.models import StandardBKT
from stanbkt.utils import VerbosityLevel

fit_opts = MCMCFitOptions(seed=1234, iter_warmup=500, iter_sampling=500, show_progress=False)

population = StandardBKT(verbose=VerbosityLevel.WARN)
population.fit(data_df, stan_fit_options=fit_opts)

individual = StandardBKT(individual_initial_knowledge=True, verbose=VerbosityLevel.WARN)
individual.fit(data_df, stan_fit_options=fit_opts)

23:40:29 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 2 had 1 divergent transitions (0.2%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


StandardBKT(fit_method=<FitMethod.MCMC: 'mcmc'>, verbose=<VerbosityLevel.WARN: 1>, is_fitted=True)

## Per student estimates

With individual initial knowledge, `pi_know` has one row per student, labeled with the student ID.

In [4]:
pi_know = individual.summary().loc["kc_0"].loc["pi_know"][["Mean", "2.5%", "97.5%"]]
pi_know.head()

Mean      2.5%     97.5%
axis       label                                     
student_id expert_stu_0  0.753293  0.001079  0.999933
           expert_stu_1  0.763708  0.002329  0.999940
           expert_stu_2  0.742590  0.001371  0.999968
           expert_stu_3  0.670388  0.000147  0.999950
           expert_stu_4  0.763304  0.013935  0.999939

In [5]:
cohort = pi_know.index.get_level_values("label").str.split("_").str[0]
pi_know.groupby(cohort.values)["Mean"].describe()[["mean", "min", "max"]]

,mean,min,max
expert,0.634721,0.103077,0.764441
novice,0.214722,0.106096,0.781714


Each student's interval is wide: a dozen responses say little about where one student started, and with `learn` in play an early wrong answer followed by right answers fits either starting state. Still, the posterior means separate the two cohorts on average. For comparison, the population model has to describe everyone with a single value:

In [6]:
population.summary().loc["kc_0"].loc["pi_know"][["Mean", "2.5%", "97.5%"]]

,,Mean,2.5%,97.5%
axis,label,,,
group_id,1,0.511074,0.344457,0.668544


## Predicting unseen students

Each `pi_know` belongs to a fitted student, so predictions are only possible for those students. Students that were not part of the fit are dropped with a warning. If none of the students were fitted, a `ValueError` is raised.

In [7]:
new_students = sim_simple_BKT(n_students=3, n_problems=12, prior=0.5, rng_seed=23, **shared)
new_students["student_id"] = "new_" + new_students["student_id"]
mixed = pd.concat([data_df[data_df["student_id"].isin(["novice_stu_0", "expert_stu_0"])], new_students])

preds = individual.predict(mixed)
preds["student_id"].unique()

stanbkt/models/core/base.py:960: UserWarning: Dropping 3 student(s) that were not part of the fit: ['new_stu_0', 'new_stu_1', 'new_stu_2'].
  self._prepare_point_estimate_prediction_inputs(


['expert_stu_0', 'novice_stu_0']
Categories (2, str): ['expert_stu_0', 'novice_stu_0']

In [8]:
try:
    individual.predict(new_students)
except ValueError as e:
    print("ValueError:", e)

ValueError: None of the students in 'data' were part of the fit. Individualized initial knowledge with CORRECTNESS_ONLY can only predict fitted students.


To predict new students while still modeling individual differences, use the JOINT strategy with student covariates, see the next example.